# Estatuto Comun

In [58]:
base = "/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral"

In [59]:
import os

paths = [i for i in os.listdir(base) if i not in [".DS_Store", "estatuto_comun"]]
print(paths)

['dpej', 'termcat', 'dictovictor', 'attentionRank', 'labourlaw', 'iate']


# ANN

In [60]:
general = []
for i in paths:
    # Obtener la lista de archivos en el directorio
    archivos = os.listdir(base+"/"+i)
    general.append({
        "path": i,
        "archivos": [i for i in archivos if not i.endswith(".conf")]
    })

In [61]:
display(general)

[{'path': 'dpej',
  'archivos': ['articulo_52_parrafo_6.ann',
   'articulo_62.ann',
   'intro.ann',
   'articulo_32_parrafo_1.ann',
   'articulo_68_parrafo_3.ann',
   'articulo_7.ann',
   'articulo_42_parrafo_2.ann',
   'articulo_42_parrafo_3.ann',
   'articulo_6.ann',
   'articulo_68_parrafo_2.ann',
   'articulo_21_parrafo_1.ann',
   'articulo_63.ann',
   'articulo_52_parrafo_7.ann',
   'articulo_36_parrafo_1.ann',
   'articulo_61.ann',
   'articulo_52_parrafo_5.ann',
   'articulo_42_parrafo_1.ann',
   'articulo_5.ann',
   'articulo_68_parrafo_1.ann',
   'articulo_52_parrafo_4.ann',
   'articulo_74.ann',
   'articulo_60.ann',
   'articulo_36_parrafo_2.ann',
   'articulo_37_parrafo_8.ann',
   'articulo_58.ann',
   'articulo_64.ann',
   'articulo_70.ann',
   'articulo_42_parrafo_4.ann',
   'articulo_56_parrafo_1.ann',
   'articulo_42_parrafo_5.ann',
   'articulo_52_parrafo_1.ann',
   'articulo_71.ann',
   'articulo_59.ann',
   'articulo_45_parrafo_1.ann',
   'articulo_73.ann',
   'artic

In [62]:
import pandas as pd

# Función para verificar si un archivo .ann tiene contenido
def check_ann_file_content(path, file):
    try:
        full_path = base+"/"+path+"/"+file
        return os.path.getsize(full_path) > 0
        
    except FileNotFoundError:
        return False

# Creación de una lista para construir el DataFrame
data_list = []

for item in general:
    path = item['path']
    ann_files = [file for file in item['archivos'] if file.endswith('.ann')]
    for ann_file in ann_files:
        has_content = check_ann_file_content(path, ann_file)
        data_list.append([path, ann_file, has_content])

In [63]:
# Creación del DataFrame
df = pd.DataFrame(data_list, columns=['path', 'ann_file', 'has_content'])
display(df)

,path,ann_file,has_content
0,dpej,articulo_52_parrafo_6.ann,True
1,dpej,articulo_62.ann,True
2,dpej,intro.ann,True
3,dpej,articulo_32_parrafo_1.ann,True
4,dpej,articulo_68_parrafo_3.ann,True
...,...,...,...
1249,iate,articulo_43_parrafo_1.ann,True
1250,iate,articulo_40_parrafo_4.ann,True
1251,iate,articulo_33_parrafo_2.ann,True
1252,iate,articulo_54_parrafo_1.ann,True


In [64]:
# Comprobamos que funciona con REBEL
labourlaw = df[df['path']=='labourlaw']
display(labourlaw)

,path,ann_file,has_content
836,labourlaw,articulo_52_parrafo_6.ann,True
837,labourlaw,articulo_62.ann,True
838,labourlaw,intro.ann,True
839,labourlaw,articulo_32_parrafo_1.ann,True
840,labourlaw,articulo_68_parrafo_3.ann,True
...,...,...,...
1040,labourlaw,articulo_43_parrafo_1.ann,True
1041,labourlaw,articulo_40_parrafo_4.ann,True
1042,labourlaw,articulo_33_parrafo_2.ann,True
1043,labourlaw,articulo_54_parrafo_1.ann,True


In [65]:
import random

# Filtramos solo las filas donde has_content es True
df_with_content = df[df['has_content'] == True]

# Verificamos si hay archivos .ann comunes entre los distintos paths
common_files = df_with_content['ann_file'].value_counts()
common_files = common_files[common_files == df_with_content['path'].nunique()]

In [66]:
display(common_files)

ann_file
articulo_52_parrafo_6.ann     6
articulo_38_parrafo_1.ann     6
articulo_14_parrafo_1.ann     6
articulo_94_parrafo_12.ann    6
articulo_94_parrafo_13.ann    6
                             ..
articulo_48_parrafo_4.ann     6
articulo_11_parrafo_3.ann     6
articulo_11_parrafo_7.ann     6
articulo_12_parrafo_2.ann     6
articulo_37_parrafo_3.ann     6
Name: count, Length: 209, dtype: int64

In [67]:
# Si existen archivos comunes, seleccionamos uno al azar
if not common_files.empty:
    selected_file = random.choice(common_files.index.tolist())
else:
    selected_file = None  # No hay archivos comunes con contenido

print(selected_file)

articulo_94_parrafo_3.ann


In [68]:
def carga_ann(ann_file):
    annotations = []
    try:
        with open(ann_file, 'r') as file:
            for line in file:
                annotations.append(line.strip())  # Agrega cada línea al array de annotations
    except FileNotFoundError:
        print(f"Archivo {ann_file} no encontrado.")
    return annotations


In [69]:
capas = df["path"].unique()
print(capas)

['dpej' 'termcat' 'dictovictor' 'attentionRank' 'labourlaw' 'iate']


In [70]:
# Diccionario para los sufijos
suffixes = {
    'dpej': "DPEJ",
    'termcat': "Termcat",
    'dictovictor': "DV",
    'attentionRank': "AttentionRank", 
    'labourlaw': "LabourLaw",
    'iate': "IATE" 
}
print(suffixes)

# Archivo donde se guardarán todas las anotaciones combinadas
output_path = base+"/estatuto_comun/"
output_file = output_path+selected_file
print(output_file)

{'dpej': 'DPEJ', 'termcat': 'Termcat', 'dictovictor': 'DV', 'attentionRank': 'AttentionRank', 'labourlaw': 'LabourLaw', 'iate': 'IATE'}
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun/articulo_94_parrafo_3.ann


In [71]:
#---------- PARA UN UNICO ARCHIVO



# terms = []       # Lista para almacenar términos (inicia con 'T')
# relations = []   # Lista para almacenar relaciones (inicia con 'R')

# # Contadores para numerar términos y relaciones de manera única
# term_counter = 1
# relation_counter = 1


# # Crea el archivo de salida

# # Recorre cada item en el diccionario 'general'
# for item in general:
#     path = item['path']
#     ann_files = [file for file in item['archivos'] if file == selected_file]
    
#     # Determina el sufijo para las anotaciones en función del path
#     suffix = suffixes.get(path, "")

#     for ann_file in ann_files:
#         full_path = base+"/"+path+"/"+ann_file
#         annotations = carga_ann(full_path)
        
#          # Modifica y guarda cada línea de anotación según su tipo
#         for annotation in annotations:
#             parts = annotation.split()
#             if annotation.startswith("T"):  # Si es un término
#                 label = parts[1] + f"-{suffix}"  # Añade el sufijo a la etiqueta
#                 modified_annotation = f"T{term_counter} {label} " + " ".join(parts[2:])
#                 terms.append(modified_annotation)
#                 term_counter += 1
#             elif annotation.startswith("R"):  # Si es una relación
#                 label = parts[1] + f"-{suffix}"  # Añade el sufijo a la etiqueta
#                 modified_annotation = f"R{relation_counter} {label} " + " ".join(parts[2:])
#                 relations.append(modified_annotation)
#                 relation_counter += 1
#             else:
#                 # Otras líneas que no comiencen con T o R (si existen) se guardan sin cambios
#                 terms.append(annotation)

# # Escribe todos los términos y luego todas las relaciones en el archivo de salida
# with open(output_file, 'w') as output:
#     for term in terms:
#         output.write(term + '\n')
#     for relation in relations:
#         output.write(relation + '\n')

# print(f"Archivo combinado guardado en {output_file} con numeración correcta y organización de términos y relaciones.")

In [72]:
# import os

#---------- PROBLEMAS DE TABULACION
# Nota: podria usar directamente df.

# def procesar_anotaciones(general, selected_file, base, suffixes, output_file):
#     """
#     Procesa anotaciones de archivos .ann y las guarda en un archivo de salida,
#     numerando correctamente términos y relaciones.

#     :param general: Lista de diccionarios con información de paths y archivos.
#     :param selected_file: Nombre del archivo .ann a procesar.
#     :param base: Ruta base donde se encuentran los archivos.
#     :param suffixes: Diccionario que asigna sufijos según el path.
#     :param output_file: Nombre del archivo donde se guardará la salida.
#     """
#     terms = []       # Lista para almacenar términos (inicia con 'T')
#     relations = []   # Lista para almacenar relaciones (inicia con 'R')

#     # Contadores para numerar términos y relaciones de manera única
#     term_counter = 1
#     relation_counter = 1

#     # Recorre cada item en el diccionario 'general'
#     for item in general:
#         path = item['path']
#         ann_files = [file for file in item['archivos'] if file == selected_file]
        
#         # Determina el sufijo para las anotaciones en función del path
#         suffix = suffixes.get(path, "")

#         for ann_file in ann_files:
#             full_path = os.path.join(base, path, ann_file)
#             annotations = carga_ann(full_path)  # Función para cargar el contenido del archivo .ann
            
#             # Modifica y guarda cada línea de anotación según su tipo
#             for annotation in annotations:
#                 parts = annotation.split()
#                 if annotation.startswith("T"):  # Si es un término
#                     if parts[1] == "concept":
#                         label = "Term" + f"-{suffix}"  # Añade el sufijo a la etiqueta
#                         modified_annotation = f"T{term_counter} {label} " + " ".join(parts[2:]) #PROBLEMA!!!!!
#                         terms.append(modified_annotation)
#                         term_counter += 1
#                     else:
#                         label = parts[1] + f"-{suffix}"  # Añade el sufijo a la etiqueta
#                         modified_annotation = f"T{term_counter} {label} " + " ".join(parts[2:]) #PROBLEMA!!!!!
#                         terms.append(modified_annotation)
#                         term_counter += 1
#                 elif annotation.startswith("R"):  # Si es una relación
#                     label = parts[1] + f"-{suffix}"  # Añade el sufijo a la etiqueta
#                     modified_annotation = f"R{relation_counter} {label} " + " ".join(parts[2:]) #PROBLEMA!!!!!
#                     relations.append(modified_annotation)
#                     relation_counter += 1
#                 else:
#                     # Otras líneas que no comiencen con T o R (si existen) se guardan sin cambios
#                     terms.append(annotation)

#     # Escribir todos los términos y luego todas las relaciones en el archivo de salida
#     with open(output_file, 'w', encoding='utf-8') as output:
#         for term in terms:
#             output.write(term + '\n')
#         for relation in relations:
#             output.write(relation + '\n')

#     print(f"Archivo combinado guardado en {output_file} con numeración correcta y organización de términos y relaciones.")


In [73]:
#---------- SOLUCION DE TABULACION Y CAMBIO DE CONCEPT A TERM


# import os

# def procesar_anotaciones(general, selected_file, base, suffixes, output_file):
#     """
#     Procesa anotaciones de archivos .ann y las guarda en un archivo de salida,
#     numerando correctamente términos y relaciones.

#     :param general: Lista de diccionarios con información de paths y archivos.
#     :param selected_file: Nombre del archivo .ann a procesar.
#     :param base: Ruta base donde se encuentran los archivos.
#     :param suffixes: Diccionario que asigna sufijos según el path.
#     :param output_file: Nombre del archivo donde se guardará la salida.
#     """
#     terms = []       # Lista para almacenar términos (inicia con 'T')
#     relations = []   # Lista para almacenar relaciones (inicia con 'R')

#     # Contadores para numerar términos y relaciones de manera única
#     term_counter = 1
#     relation_counter = 1

#     # Recorre cada item en el diccionario 'general'
#     for item in general:
#         path = item['path']
#         ann_files = [file for file in item['archivos'] if file == selected_file]
        
#         # Determina el sufijo para las anotaciones en función del path
#         suffix = suffixes.get(path, "")

#         for ann_file in ann_files:
#             full_path = os.path.join(base, path, ann_file)
#             annotations = carga_ann(full_path)  # Función para cargar el contenido del archivo .ann
            
#             # Modifica y guarda cada línea de anotación según su tipo
#             for annotation in annotations:
#                 parts = annotation.split()
                
#                 # ----- Si es un TÉRMINO (empieza con "T") -----
#                 if annotation.startswith("T"):
#                     # El label depende de si parts[1] == "concept"
#                     if parts[1] == "concept":
#                         label = "Term" + f"-{suffix}"
#                     else:
#                         label = parts[1] + f"-{suffix}"
                    
#                     # Formato BRAT para términos:
#                     # T<ID> <LABEL> <START> <END> <TEXTO>
#                     # parts: [ "TX", "concept", "144", "156", ... texto... ]
#                     start = parts[2]
#                     end = parts[3]
#                     text = " ".join(parts[4:])  # El resto se considera el texto del span
                    
#                     modified_annotation = f"T{term_counter}	{label} {start} {end}	{text}"
#                     terms.append(modified_annotation)
#                     term_counter += 1
                
#                 # ----- Si es una RELACIÓN (empieza con "R") -----
#                 elif annotation.startswith("R"):
#                     # Ejemplo BRAT: R1 <LABEL> Arg1:T2 Arg2:T3 ...
#                     label = parts[1] + f"-{suffix}"
#                     args = " ".join(parts[2:])  # Arg1:T2 Arg2:T3 ...
                    
#                     modified_annotation = f"R{relation_counter}	{label} {args}"
#                     relations.append(modified_annotation)
#                     relation_counter += 1
                
#                 # ----- Otros tipos de líneas (no empiezan con T o R) se copian igual -----
#                 else:
#                     terms.append(annotation)

#     # Escribir todos los términos y luego todas las relaciones en el archivo de salida
#     with open(output_file, 'w', encoding='utf-8') as output:
#         for term in terms:
#             output.write(term + '\n')
#         for relation in relations:
#             output.write(relation + '\n')

#     print(f"Archivo combinado guardado en {output_file} con numeración correcta y organización de términos y relaciones.")


In [86]:
#---------- AGRUPACION DE LAS ETIQUETAS EN DICCIONARIO Y AUTOMATICO. ELIMINAR DICTOVICTOR


import os

def procesar_anotaciones(general, selected_file, base, suffixes, output_file):
    """
    Procesa anotaciones de archivos .ann y las guarda en un archivo de salida,
    numerando correctamente términos y relaciones.

    :param general: Lista de diccionarios con información de paths y archivos.
    :param selected_file: Nombre del archivo .ann a procesar.
    :param base: Ruta base donde se encuentran los archivos.
    :param suffixes: Diccionario que asigna sufijos según el path.
    :param output_file: Nombre del archivo donde se guardará la salida.
    """
    terms = []       # Lista para almacenar términos (inicia con 'T')
    relations = []   # Lista para almacenar relaciones (inicia con 'R')

    # Contadores para numerar términos y relaciones de manera única
    term_counter = 1
    relation_counter = 1
    note_counter = 1  # Para las líneas # de notas

    # Este diccionario mapeará entity_key -> t_id
    # entity_key podría ser (label, start, end, text) o como prefieras
    annotation_map = {}

    
    # Recorre cada item en el diccionario 'general'
    for item in general:
        path = item['path']
        ann_files = [file for file in item['archivos'] if file == selected_file]
        
        # Determina el sufijo para las anotaciones en función del path
        suffix = suffixes.get(path, "") # Parte derecha del diccio
        # print(f"[DEBUG] suffix='{suffix}'")


        for ann_file in ann_files:
            full_path = os.path.join(base, path, ann_file)
            annotations = carga_ann(full_path)  # Función para cargar el contenido del archivo .ann
            
            # Modifica y guarda cada línea de anotación según su tipo
            for annotation in annotations:
                parts = annotation.split()

                # ----- Si es un TÉRMINO (empieza con "T") -----
                if annotation.startswith("T"):

                    if suffix == "DV": # Eliminamos aqui
                        continue
                    
                    if suffix == "AttentionRank":
                        label_general = "automatico"
                    else:
                        label_general = "refs"
                    
                    # 2) Asignar el label (si parts[1] == "concept")
                    if parts[1] == "concept":
                        label = f"Term-{label_general}"
                    else:
                        label = f"{parts[1]}-{label_general}"
                        
                    # Formato BRAT para términos:
                    # T<ID> <LABEL> <START> <END> <TEXTO>
                    # parts: [ "TX", "concept", "144", "156", ... texto... ]
                    start = parts[2]
                    end = parts[3]
                    text = " ".join(parts[4:])  # El resto se considera el texto del span
                    
                    

                    # 4) Construir una “clave” que identifique la entidad
                    #    para ver si ya existe:
                    entity_key = (label, start, end, text)

                    if entity_key not in annotation_map:
                        # No existe esta entidad -> creamos T<ID>   
                        modified_annotation = f"T{term_counter}	{label} {start} {end}	{text}"


                    # if modified_annotation not in terms:
                        terms.append(modified_annotation)

                        annotation_map[entity_key] = term_counter
                        
                        # ----- Añadir el label dentro de notas -----
                        # Ej: #1	AnnotatorNotes T2 "IATE"
                        note_line = f"#{note_counter}	AnnotatorNotes T{term_counter}	\"{suffix}\""
                        terms.append(note_line)
                        note_counter += 1

                        term_counter += 1
                    else:
                        # Si la entidad ya existe, recuperamos su T<ID>
                        existing_tid = annotation_map[entity_key]

                        # ----- Añadir el label dentro de notas -----
                        # Ej: #1	AnnotatorNotes T2 "IATE"
                        note_line = f"#{note_counter}	AnnotatorNotes T{existing_tid}	\"{suffix}\""
                        terms.append(note_line)
                        note_counter += 1
                
                # ----- Si es una RELACIÓN (empieza con "R") ----- De momento esta parte no se utiliza, pero la podemos trillar como la de arriba
                elif annotation.startswith("R"):
                    # Ejemplo BRAT: R1 <LABEL> Arg1:T2 Arg2:T3 ...
                    label = parts[1] + f"-{suffix}"
                    args = " ".join(parts[2:])  # Arg1:T2 Arg2:T3 ...
                    
                    modified_annotation = f"R{relation_counter}	{label} {args}"
                    relations.append(modified_annotation)
                    relation_counter += 1
                
                # ----- Otros tipos de líneas (no empiezan con T o R) se copian igual -----
                else:
                    terms.append(annotation)

    # Escribir todos los términos y luego todas las relaciones en el archivo de salida
    with open(output_file, 'w', encoding='utf-8') as output:
        for term in terms:
            output.write(term + '\n')
        for relation in relations:
            output.write(relation + '\n')

    print(f"Archivo combinado guardado en {output_file} con numeración correcta y organización de términos y relaciones.")


In [87]:
procesar_anotaciones(general, selected_file, base, suffixes, output_file)

Archivo combinado guardado en /Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun/articulo_94_parrafo_3.ann con numeración correcta y organización de términos y relaciones.


In [88]:
display(df)

,path,ann_file,has_content,selected_file,output_file
0,dpej,articulo_52_parrafo_6.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
1,dpej,articulo_62.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
2,dpej,intro.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
3,dpej,articulo_32_parrafo_1.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
4,dpej,articulo_68_parrafo_3.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
...,...,...,...,...,...
1249,iate,articulo_43_parrafo_1.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
1250,iate,articulo_40_parrafo_4.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
1251,iate,articulo_33_parrafo_2.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
1252,iate,articulo_54_parrafo_1.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...


In [89]:
def preparar_rutas(df, base_path, output_path):
    """
    Para cada fila del DataFrame, construye dos columnas:
    - 'selected_file': base_path + path + ann_file
    - 'output_file':   output_path + ann_file
    """
    # Crear la columna 'selected_file'
    df["selected_file"] = df.apply(
        lambda row: os.path.join(base_path, row["path"], row["ann_file"]), 
        axis=1
    )

    # Crear la columna 'output_file'
    df["output_file"] = df.apply(
        lambda row: os.path.join(output_path, row["ann_file"]), 
        axis=1
    )

    return df

In [90]:
df = preparar_rutas(df, base, output_path)
display(df)

,path,ann_file,has_content,selected_file,output_file
0,dpej,articulo_52_parrafo_6.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
1,dpej,articulo_62.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
2,dpej,intro.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
3,dpej,articulo_32_parrafo_1.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
4,dpej,articulo_68_parrafo_3.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
...,...,...,...,...,...
1249,iate,articulo_43_parrafo_1.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
1250,iate,articulo_40_parrafo_4.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
1251,iate,articulo_33_parrafo_2.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...
1252,iate,articulo_54_parrafo_1.ann,True,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...,/Users/jpardo/Desktop/Proyectos/TeresIA/Desarr...


In [91]:
for _, row in df.iterrows():
    # Extraer solo el nombre .ann
    ann_file_name = os.path.basename(row["selected_file"])  # "articulo_52_parrafo_6.ann"
    
    # Obtener la carpeta base (si tu 'procesar_anotaciones' la usa)
    # base_folder = os.path.dirname(row["selected_file"])     # "/ruta/base/termcat"

    # Llamar a la función original
    procesar_anotaciones(
        general=general, 
        selected_file=ann_file_name, 
        base=base, 
        suffixes=suffixes, 
        output_file=row["output_file"]
    )


Archivo combinado guardado en /Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun/articulo_52_parrafo_6.ann con numeración correcta y organización de términos y relaciones.
Archivo combinado guardado en /Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun/articulo_62.ann con numeración correcta y organización de términos y relaciones.
Archivo combinado guardado en /Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun/intro.ann con numeración correcta y organización de términos y relaciones.
Archivo combinado guardado en /Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun/articulo_32_parrafo_1.ann con numeración correcta y organización de términos y relaciones.
Archivo combinado guardado en /Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/es

# CONF

## annotation

In [92]:
import os

def merge_annotation_conf(files, output_file):
    """
    Combina varios archivos de configuración (annotation.conf) en uno solo.

    :param files: Lista con las rutas de los archivos .conf que quieres fusionar.
    :param output_file: Nombre del archivo de salida (por ejemplo: 'annotation.conf').
    """
    # Estructuras para almacenar datos de cada sección
    sections = {
        "entities": set(),
        "attributes": set(),
        "relations": set(),
        "events": set(),

        # Agrega más secciones si las necesitas,
        # por ejemplo: "attributes", "equivs", etc.
    }

    # Sección actual durante la lectura
    current_section = None

    for conf_file in files:
        if not os.path.exists(conf_file):
            print(f"El archivo {conf_file} no existe. Se omite.")
            continue

        with open(conf_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue  # Saltar líneas vacías

                # Detectar sección
                if line.startswith("[") and line.endswith("]"):
                    # Quitar corchetes para obtener el nombre
                    section_name = line[1:-1].lower()
                    # Actualizar sección actual si está en el diccionario
                    if section_name in sections.keys():
                        current_section = section_name
                    else:
                        # Si encuentras secciones no reconocidas, podrías
                        # crear una nueva entrada en `sections` o ignorarla
                        current_section = None
                else:
                    # Si estamos en una sección conocida, añadir la línea
                    if current_section in sections:
                        sections[current_section].add(line)

    # Crear o sobrescribir el archivo de salida
    with open(output_file, "w", encoding="utf-8") as out:
        # Recorrer cada sección en orden
        for section_name, items in sections.items():
            # Escribir cabecera de sección
            out.write(f"[{section_name}]\n")
            # Ordenar items para que sea reproducible
            for item in sorted(items):
                out.write(item + "\n")
            out.write("\n")  # Salto de línea tras cada sección

    print(f"Se han combinado los archivos en '{output_file}'.")

In [93]:
files_to_merge = []
for i in capas:
    conf1 = base+"/"+i+"/annotation.conf"
    print(conf1)
    files_to_merge.append(conf1)


/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/dpej/annotation.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/termcat/annotation.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/dictovictor/annotation.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/attentionRank/annotation.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/labourlaw/annotation.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/iate/annotation.conf


In [94]:
conf1_path = output_path+"/annotation.conf"
merge_annotation_conf(files_to_merge, conf1_path)


Se han combinado los archivos en '/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun//annotation.conf'.


In [95]:
import os

def actualizar_annotation_conf(ann_file, annotation_conf_file):
    """
    Actualiza el archivo annotation_conf_file agregando etiquetas
    (entities y relations) encontradas en ann_file.

    :param ann_file: Ruta a un archivo .ann que contiene anotaciones combinadas.
    :param annotation_conf_file: Ruta al archivo annotation.conf ya fusionado
                                 que queremos actualizar con nuevas etiquetas.
    """
    # 1) Extraer nuevas entidades y relaciones del .ann
    new_entities = set()
    new_relations = set()

    if not os.path.exists(ann_file):
        print(f"El archivo .ann '{ann_file}' no existe. Se omite actualización.")
        return

    with open(ann_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # T<id> <label> <start> <end> <texto>
            if line.startswith("T"):
                parts = line.split()
                if len(parts) > 1:
                    label = parts[1]
                    new_entities.add(label)

            # R<id> <label> Arg1:T# Arg2:T# ...
            elif line.startswith("R"):
                parts = line.split()
                if len(parts) > 1:
                    label = parts[1]
                    new_relations.add(label)

    # Si no hay nada nuevo, podemos opcionalmente abortar
    if not new_entities and not new_relations:
        print("No se encontraron nuevas entidades o relaciones en el archivo .ann.")
        return

    # 2) Leer y parsear el annotation.conf final (ya fusionado)
    sections = {
        "entities": set(),
        "attributes": set(),
        "relations": set(),
        "events": set(),
    }
    current_section = None

    if not os.path.exists(annotation_conf_file):
        print(f"El archivo de configuración '{annotation_conf_file}' no existe.")
        return

    with open(annotation_conf_file, "r", encoding="utf-8") as cf:
        for line in cf:
            line_stripped = line.strip()
            if not line_stripped:
                continue

            # Detectar [secciones]
            if line_stripped.startswith("[") and line_stripped.endswith("]"):
                sec_name = line_stripped[1:-1].lower()  # "entities", "relations", etc.
                if sec_name in sections:
                    current_section = sec_name
                else:
                    current_section = None
            else:
                # Guardar la línea en la sección actual (si es válida)
                if current_section in sections:
                    sections[current_section].add(line_stripped)

    # 3) Actualizar secciones con las nuevas etiquetas
    #    (suponemos que todas las etiquetas "T" van a entities y "R" van a relations)
    sections["entities"].update(new_entities)
    sections["relations"].update(new_relations)

    # 4) Reescribir el annotation.conf actualizado
    with open(annotation_conf_file, "w", encoding="utf-8") as out:
        for sec, items in sections.items():
            out.write(f"[{sec}]\n")
            for item in sorted(items):
                out.write(item + "\n")
            out.write("\n")

    print(f"Archivo '{annotation_conf_file}' actualizado con nuevas etiquetas de '{ann_file}'.")


In [96]:
# Actualizar annotation.conf final con las nuevas etiquetas detectadas en el .ann:
actualizar_annotation_conf("/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun/articulo_1_parrafo_1.ann", conf1_path)


Archivo '/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun//annotation.conf' actualizado con nuevas etiquetas de '/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun/articulo_1_parrafo_1.ann'.


## tools

In [97]:
import os

import os

def merge_tools_conf(files, output_file):
    """
    Combina varios archivos tools.conf en uno solo,
    unificando solo la sección [options] sin duplicados.
    
    :param files: Lista de rutas de archivos .conf que quieres fusionar.
    :param output_file: Nombre (o ruta) del archivo de salida resultante, ej. 'tools.conf'.
    """
    # Usamos un set para no duplicar opciones
    options_set = set()

    current_section = None

    for conf_file in files:
        if not os.path.exists(conf_file):
            print(f"El archivo {conf_file} no existe. Se omite.")
            continue

        with open(conf_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                # Saltar líneas vacías
                if not line:
                    continue

                # Detectar si empieza sección [options]
                if line.startswith("[") and line.endswith("]"):
                    section_name = line[1:-1].lower()
                    # Si encontramos [options], cambiamos current_section
                    if section_name == "options":
                        current_section = "options"
                    else:
                        current_section = None
                else:
                    # Si estamos en la sección [options], añadimos la línea al set
                    if current_section == "options":
                        options_set.add(line)

    # Escribir archivo final con la sección [options]
    with open(output_file, "w", encoding="utf-8") as out:
        out.write("[options]\n")
        # Ordenar las líneas para tener consistencia
        for item in sorted(options_set):
            out.write(item + "\n")

    print(f"Se han combinado los archivos en '{output_file}'.")


In [98]:
files_to_merge = []
for i in capas:
    conf3 = base+"/"+i+"/tools.conf"
    print(conf3)
    files_to_merge.append(conf3)

/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/dpej/tools.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/termcat/tools.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/dictovictor/tools.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/attentionRank/tools.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/labourlaw/tools.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/iate/tools.conf


In [99]:
conf2_path = output_path+"/tools.conf"
merge_tools_conf(files_to_merge, conf2_path)


Se han combinado los archivos en '/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun//tools.conf'.


## visual

In [100]:
import os

def merge_visual_conf(files, output_file):
    """
    Combina varios archivos visual.conf en uno solo,
    unificando sus secciones [labels] y [drawing] sin duplicados.
    
    :param files: Lista de rutas de archivos .conf que quieres fusionar.
    :param output_file: Nombre (o ruta) del archivo de salida resultante, ej. 'visual.conf'.
    """
    # Diccionario para almacenar cada sección en un set (evitar duplicados)
    sections = {
        "labels": set(),
        "drawing": set()
    }

    current_section = None

    for conf_file in files:
        if not os.path.exists(conf_file):
            print(f"El archivo {conf_file} no existe. Se omite.")
            continue

        with open(conf_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                # Omitir líneas vacías
                if not line:
                    continue

                # Detectar cabecera de sección, p.ej. "[labels]", "[drawing]"
                if line.startswith("[") and line.endswith("]"):
                    section_name = line[1:-1].lower()  # convertir a minúsculas
                    if section_name in sections:
                        current_section = section_name
                    else:
                        current_section = None
                else:
                    # Si estamos dentro de una sección válida, agregamos la línea
                    if current_section in sections:
                        sections[current_section].add(line)

    # Crear o sobrescribir el archivo de salida con las secciones unificadas
    with open(output_file, "w", encoding="utf-8") as out:
        # Sección [labels]
        out.write("[labels]\n")
        for item in sorted(sections["labels"]):
            out.write(item + "\n")
        out.write("\n")

        # Sección [drawing]
        out.write("[drawing]\n")
        for item in sorted(sections["drawing"]):
            out.write(item + "\n")
        out.write("\n")

    print(f"Se han combinado los archivos en '{output_file}'.")

In [101]:
files_to_merge = []
for i in capas:
    conf3 = base+"/"+i+"/visual.conf"
    print(conf3)
    files_to_merge.append(conf3)

/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/dpej/visual.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/termcat/visual.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/dictovictor/visual.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/attentionRank/visual.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/labourlaw/visual.conf
/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/iate/visual.conf


In [102]:
conf3_path = output_path+"/visual.conf"
merge_visual_conf(files_to_merge, conf3_path)


Se han combinado los archivos en '/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun//visual.conf'.


In [103]:
import os
import random

def actualizar_visual_conf(ann_file, visual_conf_file):
    """
    Actualiza el archivo 'visual_conf_file' agregando las etiquetas
    encontradas en 'ann_file' (líneas que inicien con T) y asignando
    un color distinto a cada una en la sección [drawing].

    :param ann_file: Ruta al archivo .ann combinado (que contiene nuevas etiquetas).
    :param visual_conf_file: Ruta al archivo 'visual.conf' ya fusionado que se modificará.
    """
    # 1) Extraer las nuevas etiquetas del archivo .ann
    new_labels = set()
    if not os.path.exists(ann_file):
        print(f"El archivo .ann '{ann_file}' no existe. Se omite la actualización.")
        return

    with open(ann_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Cada línea de término: T<num> <etiqueta> <start> <end> <texto...>
            if line.startswith("T"):
                parts = line.split()
                if len(parts) > 1:
                    label = parts[1]  # La segunda palabra es la etiqueta
                    new_labels.add(label)

    if not new_labels:
        print("No se encontraron etiquetas nuevas en el archivo .ann.")
        return

    # 2) Leer y parsear el visual.conf final
    sections = {
        "labels": set(),
        "drawing": set()
    }
    current_section = None

    if not os.path.exists(visual_conf_file):
        print(f"El archivo de configuración '{visual_conf_file}' no existe.")
        return

    with open(visual_conf_file, "r", encoding="utf-8") as cf:
        for line in cf:
            line_stripped = line.strip()
            if not line_stripped:
                continue

            # Detectar [labels] o [drawing]
            if line_stripped.startswith("[") and line_stripped.endswith("]"):
                sec_name = line_stripped[1:-1].lower()  # "labels" o "drawing"
                if sec_name in sections:
                    current_section = sec_name
                else:
                    current_section = None
            else:
                if current_section in sections:
                    sections[current_section].add(line_stripped)

    # 3) Generar un color aleatorio para cada etiqueta, y actualizar las secciones
    #    Mantenemos un diccionario de label->color para ser consistente si la etiqueta aparece varias veces
    color_map = {}
    for label in new_labels:
        # Creamos una entrada en [labels]
        # Ejemplo: "concept | CONCEPT"
        sections["labels"].add(f"{label} | {label.upper()}")

        # Generar color si no existe en el map
        if label not in color_map:
            color_map[label] = "#{:06x}".format(random.randint(0, 0xFFFFFF))

        # Creamos una entrada en [drawing]
        # Ejemplo: "concept	bgColor:#ffccaa, fgColor:black, borderColor:darken"
        color_line = f"{label}\tbgColor:{color_map[label]}, fgColor:black, borderColor:darken"
        sections["drawing"].add(color_line)

    # 4) Reescribir el visual.conf actualizado
    with open(visual_conf_file, "w", encoding="utf-8") as out:
        # Sección [labels]
        out.write("[labels]\n")
        for item in sorted(sections["labels"]):
            out.write(item + "\n")
        out.write("\n")

        # Sección [drawing]
        out.write("[drawing]\n")
        for item in sorted(sections["drawing"]):
            out.write(item + "\n")
        out.write("\n")

    print(f"Archivo '{visual_conf_file}' actualizado con las nuevas etiquetas de '{ann_file}'.")


In [104]:
# Actualizar annotation.conf final con las nuevas etiquetas detectadas en el .ann:
actualizar_visual_conf("/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun/articulo_1_parrafo_1.ann", conf3_path)


Archivo '/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun//visual.conf' actualizado con las nuevas etiquetas de '/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun/articulo_1_parrafo_1.ann'.


# Renombrar

In [15]:
import pandas as pd

path = "/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/corpus/anotacion/parsing.csv"
parsing = pd.read_csv(path)

display(parsing)

,old,new,folder
0,requerimiento-10_parrafo_1.txt,ejemplo_0_0.txt,artText_UNED
1,resolucion-particular-9_parrafo_2.txt,ejemplo_0_1.txt,artText_UNED
2,resolucion-particular-9_parrafo_3.txt,ejemplo_0_2.txt,artText_UNED
3,requerimiento-14_parrafo_1.txt,ejemplo_0_3.txt,artText_UNED
4,acta-inspeccion-8_parrafo_1.txt,ejemplo_0_4.txt,artText_UNED
...,...,...,...
2823,d00026-plain_parrafo_20.txt,ejemplo_7_39.txt,Admin_IULA
2824,d00026-plain_parrafo_36.txt,ejemplo_7_40.txt,Admin_IULA
2825,d00026-plain_parrafo_22.txt,ejemplo_7_41.txt,Admin_IULA
2826,d00026-plain_parrafo_23.txt,ejemplo_7_42.txt,Admin_IULA


In [16]:
laboral2 = parsing[parsing['folder'] == "Laboral2_UPM"]
display(laboral2)

,old,new,folder
285,articulo_45_parrafo_1.txt,ejemplo_3_0.txt,Laboral2_UPM
286,articulo_71.txt,ejemplo_3_1.txt,Laboral2_UPM
287,articulo_52_parrafo_1.txt,ejemplo_3_2.txt,Laboral2_UPM
288,articulo_59.txt,ejemplo_3_3.txt,Laboral2_UPM
289,articulo_42_parrafo_5.txt,ejemplo_3_4.txt,Laboral2_UPM
...,...,...,...
489,articulo_83.txt,ejemplo_3_204.txt,Laboral2_UPM
490,articulo_34_parrafo_1.txt,ejemplo_3_205.txt,Laboral2_UPM
491,intro_parrafo_1.txt,ejemplo_3_206.txt,Laboral2_UPM
492,articulo_23_parrafo_1.txt,ejemplo_3_207.txt,Laboral2_UPM


In [17]:
import re
import os
import shutil
import chardet
import unicodedata
import pandas as pd


def change_extension(nombre_archivo):
    """
    Cambia la extensión de un nombre de archivo a anotacion.

    :param nombre_archivo: Nombre del archivo con extensión (ej: 'documento.txt')
    :return: Nombre del archivo sin extensión (ej: 'documento')
    """
    name = os.path.splitext(nombre_archivo)[0]
    annot = name+".ann"
    return annot

def remove_accents_and_spaces(text):
    """Elimina tildes y caracteres especiales, y reemplaza espacios con _"""
    # Elimina tildes y caracteres diacríticos
    text = ''.join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))
    # Reemplaza espacios por _
    text = text.replace(" ", "_")
    return text


def rename_files_in_folder(folder_path, dataset): # Modificada y diferente de 'seleccion.ipynb'
    """
    Recorre los archivos de una carpeta y renombra cada archivo según los nombres
    antiguos y nuevos especificados en el DataFrame 'dataset'. Se verifica que cada
    archivo (old) exista en la carpeta y es renombrado al nombre nuevo (new).

    :param folder_path: Ruta de la carpeta que contiene los archivos.
    :param dataset: DataFrame que contiene las columnas 'old' y 'new' con los nombres de archivo.
    :return: DataFrame con los registros de los archivos renombrados (old y new).
    """
    # Lista de archivos actualmente en la carpeta (solo nombres)
    archivos_old = os.listdir(folder_path)
    registros = []  # Para guardar los cambios realizados

    # Iterar sobre cada fila del dataset
    for _, row in dataset.iterrows():
        old_file = row['old']
        new_file = row['new']
        
        # Verificar si el nombre de archivo antiguo existe en la carpeta
        if old_file in archivos_old:
            # Construir la ruta completa del archivo antiguo
            old_path = os.path.join(folder_path, old_file)
            if os.path.isfile(old_path):
                # Construir la ruta completa del nuevo nombre
                new_path = os.path.join(folder_path, new_file)
                # Renombrar el archivo
                os.rename(old_path, new_path)
                print(f"Renamed: {old_file} -> {new_file}")
                registros.append({'old': old_file, 'new': new_file})
            else:
                print(f"Not a file: {old_path}")
        else:
            print(f"File not found: {old_file}")
    
    # Crear un DataFrame con los registros de los archivos renombrados
    df_registros = pd.DataFrame(registros)
    return df_registros


In [18]:
# Ahora, aplicamos change_extension a las columnas 'old' y 'new'
laboral2['old'] = laboral2['old'].apply(change_extension)
laboral2['new'] = laboral2['new'].apply(change_extension)

# Mostramos el DataFrame resultante
display(laboral2)


/var/folders/wy/5tnvpqhs3tn3h3821ywg2j7w0000gn/T/ipykernel_12331/1158868910.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  laboral2['old'] = laboral2['old'].apply(change_extension)
/var/folders/wy/5tnvpqhs3tn3h3821ywg2j7w0000gn/T/ipykernel_12331/1158868910.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  laboral2['new'] = laboral2['new'].apply(change_extension)


,old,new,folder
285,articulo_45_parrafo_1.ann,ejemplo_3_0.ann,Laboral2_UPM
286,articulo_71.ann,ejemplo_3_1.ann,Laboral2_UPM
287,articulo_52_parrafo_1.ann,ejemplo_3_2.ann,Laboral2_UPM
288,articulo_59.ann,ejemplo_3_3.ann,Laboral2_UPM
289,articulo_42_parrafo_5.ann,ejemplo_3_4.ann,Laboral2_UPM
...,...,...,...
489,articulo_83.ann,ejemplo_3_204.ann,Laboral2_UPM
490,articulo_34_parrafo_1.ann,ejemplo_3_205.ann,Laboral2_UPM
491,intro_parrafo_1.ann,ejemplo_3_206.ann,Laboral2_UPM
492,articulo_23_parrafo_1.ann,ejemplo_3_207.ann,Laboral2_UPM


In [19]:
# Supongamos que ya tienes un DataFrame, por ejemplo:
df_renamed = rename_files_in_folder("/Users/jpardo/Desktop/Proyectos/TeresIA/Desarrollo/server/gold_standard2/estatuto_laboral/estatuto_comun_renamed", laboral2)

Renamed: articulo_45_parrafo_1.ann -> ejemplo_3_0.ann
Renamed: articulo_71.ann -> ejemplo_3_1.ann
Renamed: articulo_52_parrafo_1.ann -> ejemplo_3_2.ann
Renamed: articulo_59.ann -> ejemplo_3_3.ann
Renamed: articulo_42_parrafo_5.ann -> ejemplo_3_4.ann
Renamed: articulo_42_parrafo_4.ann -> ejemplo_3_5.ann
Renamed: articulo_56_parrafo_1.ann -> ejemplo_3_6.ann
Renamed: articulo_58.ann -> ejemplo_3_7.ann
Renamed: articulo_70.ann -> ejemplo_3_8.ann
Renamed: articulo_64.ann -> ejemplo_3_9.ann
Renamed: articulo_37_parrafo_8.ann -> ejemplo_3_10.ann
Renamed: articulo_4_parrafo_1.ann -> ejemplo_3_11.ann
Renamed: articulo_45_parrafo_2.ann -> ejemplo_3_12.ann
Renamed: articulo_35_parrafo_1.ann -> ejemplo_3_13.ann
Renamed: articulo_52_parrafo_2.ann -> ejemplo_3_14.ann
Renamed: articulo_88_parrafo_1.ann -> ejemplo_3_15.ann
Renamed: articulo_3.ann -> ejemplo_3_16.ann
Renamed: articulo_48_parrafo_10.ann -> ejemplo_3_17.ann
Renamed: articulo_48_parrafo_11.ann -> ejemplo_3_18.ann
Renamed: articulo_2.ann -

In [20]:
display(df_renamed)

,old,new
0,articulo_45_parrafo_1.ann,ejemplo_3_0.ann
1,articulo_71.ann,ejemplo_3_1.ann
2,articulo_52_parrafo_1.ann,ejemplo_3_2.ann
3,articulo_59.ann,ejemplo_3_3.ann
4,articulo_42_parrafo_5.ann,ejemplo_3_4.ann
...,...,...
204,articulo_83.ann,ejemplo_3_204.ann
205,articulo_34_parrafo_1.ann,ejemplo_3_205.ann
206,intro_parrafo_1.ann,ejemplo_3_206.ann
207,articulo_23_parrafo_1.ann,ejemplo_3_207.ann
